In [ ]:
import pandas as pd
from pacti.contracts import PolyhedralIoContract
from pacti.utils.plots import plot_assumptions
from pacti.utils.plots import plot_guarantees
from pathlib import Path

#to create contracts we need the IL IH: 
#structure should be as follows for a NOR gate:
# assume (x < IL)
# then gaurentee output = ON and in range [y(IL), ymax]
#assume (x > IH)
# then gaurentee output = OFF and in range [ymin, y(IH)]
#assume (IL < x < IH) 
# then gaurentee output = transition  and in range [y(IH), y(IL)]

csv_files = [
    "/home/shrut/bioCircuitProject/library/NOR_gate/parsed_Eco1C1G1T1.csv",
    "/home/shrut/bioCircuitProject/library/NOR_gate/parsed_Eco1C1G1T1b.csv", 
    "/home/shrut/bioCircuitProject/library/NOR_gate/parsed_EcoCB1_v1.csv",
    "/home/shrut/bioCircuitProject/library/NOR_gate/parsed_EcoCB1_v2.csv",
    "/home/shrut/bioCircuitProject/library/NOR_gate/parsed_EcoCB1.csv",
    "/home/shrut/bioCircuitProject/library/NOR_gate/parsed_EcoJS4ib_110215.csv",
    "/home/shrut/bioCircuitProject/library/NOR_gate/parsed_EcoJS4ib_111015.csv",
    "/home/shrut/bioCircuitProject/library/NOR_gate/parsed_EcoJS4ib.csv",
]

csv_files_names = [
    "parsed_Eco1C1G1T1.csv",
    "parsed_Eco1C1G1T1b.csv",
    "parsed_EcoCB1_v1.csv",
    "parsed_EcoCB1_v2.csv",
    "parsed_EcoCB1.csv",
    "parsed_EcoJS4ib_110215.csv",
    "parsed_EcoJS4ib_111015.csv",
    "parsed_EcoJS4ib.csv",
]

# df = pd.read_csv(csv_files[0])

#to be able to figure out corresponding y values for given x values (IL and IH are x values)
#also important because contracts say assuming x is in a certain range, then y is guaranteed to be in a certain range 
def corresponding_y(x, row):
    ymin = float(row["ymin"])
    ymax = float(row["ymax"])
    k = float(row["K"])
    n = float(row["n"])
    return ymin + (ymax - ymin) / (1.0 + (x / k) ** n)

def make_nor_contracts(row):
    #contracts are based on these two x - values 
    il = float(row["IL"])
    ih = float(row["IH"])

    if il >= ih:
        raise ValueError(f"Expected IL < IH; got IL={il}, IH={ih}")

    #need corresponding y values for the regions defined by IL and IH (for the gaurentee part of the contract)

    y_at_il = corresponding_y(il, row)
    y_at_ih = corresponding_y(ih, row)
    ymin = float(row["ymin"])
    ymax = float(row["ymax"])



    # [y(IL), ymax] -> range of y values for: x < IL -> ON
    # [y(IL), y(IH)] -> range of y values for: IL < x < IH -> transition
    # [ymin, y(IH)] -> range of y values for: x > IH -> OFF

    #returns dictionary of three contracts PER row 
    return {
        "ON": PolyhedralIoContract.from_strings(
            input_vars=["x"],
            output_vars=["y"],

            assumptions=["-x <= 0", f"x <= {il:.12g}"], #negating both sides reverses ineqality 
            guarantees=[f"-y <= {-y_at_il:.12g}", f"y <= {ymax:.12g}"],
        ),
        "transition": PolyhedralIoContract.from_strings(
            input_vars=["x"],
            output_vars=["y"],
            assumptions=[f"-x <= {-il:.12g}", f"x <= {ih:.12g}"],
            guarantees=[f"-y <= {-y_at_ih:.12g}", f"y <= {y_at_il:.12g}"],
        ),
        "OFF": PolyhedralIoContract.from_strings(
            input_vars=["x"],
            output_vars=["y"],
            assumptions=[f"-x <= {-ih:.12g}"],
            guarantees=[f"-y <= {-ymin:.12g}", f"y <= {y_at_ih:.12g}"],
        ),
    }

#organizing dictionary by file name and protien name 
nor_contracts = {}
for file_name, csv_path in zip(csv_files_names, csv_files):
    nor_contracts[file_name] = {}
    df = pd.read_csv(csv_path)
    for i, row in df.iterrows():
        protein_name = str(row["name"])
        # need to double check that none of the files have multiple graphs w same protien name not sure if itll overwrite 
        # if protein_name in nor_contracts[file_name]:
        #     print(f" Multiple entries found for protein '{protein_name}' in file '{file_name}'")
        nor_contracts[file_name][protein_name] = make_nor_contracts(row)
        #check if the contracts are correct by using plot_assumptions and plot_garantees functions
        x_max = (2.0 * float(row["IH"]))  # setting x_max to twice the IH value for plotting 
        plot_assumptions(contract=nor_contracts[file_name][protein_name]["ON"], x_var="x", y_var="y",var_values={}, x_lims=(0.0, x_max),y_lims=(float(row["ymin"]), float(row["ymax"])),show=True,)
        plot_guarantees(contract=nor_contracts[file_name][protein_name]["ON"], x_var="x", y_var="y",var_values={}, x_lims=(0.0, x_max),y_lims=(float(row["ymin"]), float(row["ymax"])),show=True,)
        
        plot_assumptions(contract=nor_contracts[file_name][protein_name]["OFF"], x_var="x", y_var="y",var_values={}, x_lims=(0.0, x_max),y_lims=(float(row["ymin"]), float(row["ymax"])),show=True,)
        plot_guarantees(contract=nor_contracts[file_name][protein_name]["OFF"], x_var="x", y_var="y",var_values={}, x_lims=(0.0, x_max),y_lims=(float(row["ymin"]), float(row["ymax"])),show=True,)

        plot_assumptions(contract=nor_contracts[file_name][protein_name]["transition"], x_var="x", y_var="y",var_values={}, x_lims=(0.0, x_max),y_lims=(float(row["ymin"]), float(row["ymax"])),show=True,)
        plot_guarantees(contract=nor_contracts[file_name][protein_name]["transition"], x_var="x", y_var="y",var_values={}, x_lims=(0.0, x_max),y_lims=(float(row["ymin"]), float(row["ymax"])),show=True,)

# for i in range(len(df)):
#     protein_name = str(df.iloc[i]["name"])
#     print(nor_contracts[protein_name]["ON"])
#     print(nor_contracts[protein_name]["OFF"])
#     print(nor_contracts[protein_name]["transition"])

In [ ]:
# Five contracts are returned from the AND gate:
# ON: theres only one condition that creates this 
# If x1 > IH1 AND x2 > IH2, then y is guaranteed to be in [y(IH1, IH2), ymax]
# OFF: multiple conditions can create this
# If x1 < IL1 OR x2 < IL2, then the output is OFF
# OFF1: assuming x1 < IL1 guarantees y is in [ymin, y(IL1, ymax)].
# OFF2: assuming x2 < IL2 guarantees y is in [ymin, y(ymax, IL2)].
# TRANSITION:
# If IL1 < x1 < IH1 OR IL2 < x2 < IH2,
# transition1:assuming IL1 < x1 < IH1 AND x2 > IL2 guarantees y is in [y(IL1, IL2), y(IH1, ymax)].
# transition2:assuming x1 > IL1 AND IL2 < x2 < IH2 guarantees y is in [y(IL1, IL2), y(ymax, IH2)].


#should have five contracts: 
    #three for OFF, one for ON, and ONE for transitional 
    #transitional: only if both inputs are in transition region 
    #ON only if both inputs are high 
    #OFF when 
    # (input1 < IL1, input2 > IL2)
    # (input2 < IL2, Input1 > IL1)
    # (input1 < IL1, input2 < IL2) both greater than low but less than high
    

csv_and_file = "/home/shrut/bioCircuitProject/library/AND_gate/parsed_Eco3C3G3T3.csv"
and_file_name = "parsed_Eco3C3G3T3.csv"

#this came from defining thresholds for AND gates in differente file 
threshold_values = pd.read_csv("/home/shrut/bioCircuitProject/library/AND_gate/and_thresholds.csv")


INF = float("inf")

#will need to know corresponding y values esp for gaurentees 
def corresponding_y_and(x1, x2, row):
    ymin = float(row["ymin"])
    ymax = float(row["ymax"])
    k1 = float(row["K1"])
    k2 = float(row["K2"])
    n = float(row["n"])

    h1 = 1.0 if x1 == INF else x1**n / (k1**n + x1**n)
    h2 = 1.0 if x2 == INF else x2**n / (k2**n + x2**n)

    return ymin + (ymax - ymin) * h1 * h2



# ON: x1 >= IH1 AND x2 >= IH2
# OFF1:
#   0 <= x1 <= IL1 AND x2 >= IL2
#
# OFF2:
#   x1 >= IL1 AND 0 <= x2 <= IL2
#
# OFF3:
#   0 <= x1 <= IL1 AND 0 <= x2 <= IL2
#
# TRANSITION:
#   IL1 <= x1 <= IH1 AND IL2 <= x2 <= IH2
#
# The output bounds assume the Cello response
# increases monotonically with both inputs.


def make_and_contracts(row, threshold_row):

    il1 = float(threshold_row["IL1"])
    ih1 = float(threshold_row["IH1"])
    il2 = float(threshold_row["IL2"])
    ih2 = float(threshold_row["IH2"])

    ymin = float(row["ymin"])
    ymax = float(row["ymax"])

    if not (0 <= il1 < ih1 and 0 <= il2 < ih2):
        raise ValueError("Expected 0 <= IL1 < IH1 and 0 <= IL2 < IH2")

    # ON: x1 >= IH1 AND x2 >= IH2 so both inptuts are high
    # this is the lower bound for gaurentees when ON 
    y_on_low = corresponding_y_and(ih1, ih2, row)

    # OFF1: x1 LOW, x2 >= IL2 
    #this is the upper bound for gaurentees when OFF1
    y_off1_high = corresponding_y_and(il1, INF, row)

    # OFF2: x2 LOW, x1 >= IL1
    #this is the upper bound for gaurentees when OFF2
    y_off2_high = corresponding_y_and(INF, il2, row)

    # OFF3: both inputs LOW
    #lower bound for gaurentees when OFF
    y_off3_high = corresponding_y_and(il1, il2, row)

    # TRANSITION: only happens when both inputs between IL and IH
    # so these are both the corresponding upper and lower bounds for transition region 
    y_trans_low = corresponding_y_and(il1, il2, row)
    y_trans_high = corresponding_y_and(ih1, ih2, row)

    return {

        #On contract 
        "ON": PolyhedralIoContract.from_strings(
            input_vars=["x1", "x2"],
            output_vars=["y"],

            #assumptions: x1 >= IH1 AND x2 >= IH2 -> both inputs are high 
            assumptions=[f"-x1 <= {-ih1:.12g}", f"-x2 <= {-ih2:.12g}" ],

            #if this assumption is true, then y is guaranteed to be in [y(IH1, IH2), ymax]
            guarantees=[f"-y <= {-y_on_low:.12g}", f"y <= {ymax:.12g}"],
        ),

        #first off contract 
        "OFF1": PolyhedralIoContract.from_strings(
            input_vars=["x1", "x2"],
            output_vars=["y"],

            #assumptions: x1 < IL1 AND x2 >= IL2 -> x1 is low, x2 is high
            assumptions=["-x1 <= 0",f"x1 <= {il1:.12g}",f"-x2 <= {-il2:.12g}"],

            #considering the assumption, y is guaranteed to be in [ymin, y(IL1, ymax)]
            guarantees=[f"-y <= {-ymin:.12g}", f"y <= {y_off1_high:.12g}"],
        ),

        #second off contract 
        "OFF2": PolyhedralIoContract.from_strings(
            input_vars=["x1", "x2"],
            output_vars=["y"],

            #assuming x2 < IL2 AND x1 >= IL1 -> x2 is low, x1 is high
            assumptions=[f"-x1 <= {-il1:.12g}","-x2 <= 0",f"x2 <= {il2:.12g}"],

            #then we can gaurentee that y is in [ymin, y(ymax, IL2)]
            guarantees=[ f"-y <= {-ymin:.12g}",f"y <= {y_off2_high:.12g}"],
        ),

        #last off contract 
        "OFF3": PolyhedralIoContract.from_strings(
            input_vars=["x1", "x2"],
            output_vars=["y"],

            #assuming both inputs are low
            assumptions=["-x1 <= 0",f"x1 <= {il1:.12g}","-x2 <= 0", f"x2 <= {il2:.12g}"],

            #then we can gaurentee that y is in [ymin, y(IL1, IL2)]
            guarantees=[f"-y <= {-ymin:.12g}",f"y <= {y_off3_high:.12g}"],
        ),

        #transition contract 
        "transition": PolyhedralIoContract.from_strings(
            input_vars=["x1", "x2"],
            output_vars=["y"],

            #assuming both inputs are in transition region -> (IL1 < x1 < IH1 AND IL2 < x2 < IH2)
            assumptions=[f"-x1 <= {-il1:.12g}", f"x1 <= {ih1:.12g}", f"-x2 <= {-il2:.12g}", f"x2 <= {ih2:.12g}"],

            #then we can gaurentee that y is in [y(IL1, IL2), y(IH1, IH2)]
            guarantees=[f"-y <= {-y_trans_low:.12g}", f"y <= {y_trans_high:.12g}"],
        ),
    }


#creating contracts: 

and_df = pd.read_csv(csv_and_file)
and_df = and_df[and_df["type"] == "AND"].reset_index(drop=True)

and_contracts = {and_file_name: {}}

#for each row we make five contracts 
for _, row in and_df.iterrows():

    protein_name = str(row["name"])

    match = threshold_values[threshold_values["name"] == protein_name]

    if match.empty:
        print(f"No thresholds found for {protein_name}")
        continue

    threshold_row = match.iloc[0]

    and_contracts[and_file_name][protein_name] = (make_and_contracts(row, threshold_row))

    # Thresholds for plotting values comes from graphingAND.ipynb
    il1 = float(threshold_row["IL1"])
    ih1 = float(threshold_row["IH1"])
    il2 = float(threshold_row["IL2"])
    ih2 = float(threshold_row["IH2"])

    x1_max = 2.0 * ih1
    x2_max = 2.0 * ih2

    y_lims = (float(row["ymin"]), float(row["ymax"]))

    contracts = and_contracts[and_file_name][protein_name]

    #this is for plotting the x1 gaurentees and assumptions, we fix x2 be values in each assumption region 
    x2_values = {
        "ON": 2.0 * ih2,
        "OFF1": ih2,
        "OFF2": il2 / 2.0,
        "OFF3": il2 / 2.0,
        "transition": (il2 + ih2) / 2.0
    }

    #for plotting the x2 gaurentees and assumptions, we fix x1 be values in each assumption region
    x1_values = {
        "ON": 2.0 * ih1,
        "OFF1": il1 / 2.0,
        "OFF2": ih1,
        "OFF3": il1 / 2.0,
        "transition": (il1 + ih1) / 2.0
    }



    for contract_name in ["ON","OFF1","OFF2","OFF3","transition"]:

        contract = contracts[contract_name]

        # Plot assumptions when x2 is fixed to a value in the assumption region for each contract
        plot_assumptions(contract=contract,x_var="x1",y_var="x2",var_values={},x_lims=(0.0, x1_max),y_lims=(0.0, x2_max),show=True,)
        # Plot guarantees when x2 is fixed 
        plot_guarantees(contract=contract,x_var="x1",y_var="y", var_values={"x2": x2_values[contract_name]}, x_lims=(0.0, x1_max),y_lims=y_lims,show=True,)

        #plot assumptions when x1 is fixed
        plot_assumptions(contract=contract,x_var="x2",y_var="x1",var_values={},x_lims=(0.0, x2_max),y_lims=(0.0, x1_max),show=True,)        
        # Plot guarantees when x1 is fixed to a value in the assumption region for each contract
        plot_guarantees(contract=contract,x_var="x2",y_var="y", var_values={"x1": x1_values[contract_name]}, x_lims=(0.0, x2_max),y_lims=y_lims,show=True,)
        

In [ ]:
#creating a combinational circuit 

#need to create function that matches nor input to a contract 
def matching_to_nor_contracts(in1, row, contracts): 
    # print("Protein:", row["name"])
    # print("IL:", row["IL"])
    # print("IH:", row["IH"])
    # print("Input:", in1)
    # u1 is user input 
    # row is the row from the csv we are looking at
    # contracts is the dictionary of contracts we created earlier
    il = float(row["IL"])
    ih = float(row["IH"])

    if in1 < 0:
        raise ValueError("Input concentration shouldnt be negative")  
    if in1 <= il:
        contract_name = "ON"
    elif in1 >= ih:
        contract_name = "OFF"
    else:
        contract_name = "transition"    

    #recording the concentration of the output 
    output_concentration = corresponding_y(in1, row)

    return contracts[contract_name], contract_name, output_concentration 

#created a NOR gate that takes in input u1 and outputs the state 
file_name = "parsed_Eco1C1G1T1.csv"
csv_path = csv_files[csv_files_names.index(file_name)]
df = pd.read_csv(csv_path)
protein_name = "A1_AmtR"
row = df[df["name"] == protein_name].iloc[0]
u1 = 0.9 #my "user input" 
contracts = nor_contracts[file_name][protein_name]  #these are all the "filtered" contracts by the protien name so that we can pass into the matching function 
selected_contract, contract_name, concentration = matching_to_nor_contracts(u1, row, contracts)
print(concentration, "of", protein_name, "and the state is:", contract_name)


def matching_to_and_contracts(in1, in2, thresholds_row, row, contracts):

    # Thresholds from and_thresholds.csv
    il1 = float(threshold_row["IL1"])
    ih1 = float(threshold_row["IH1"])
    il2 = float(threshold_row["IL2"])
    ih2 = float(threshold_row["IH2"])

    if in1 < 0 or in2 < 0:
        raise ValueError("Input concentrations shouldn't be negative")

    if in1 >= ih1 and in2 >= ih2:
        contract_name = "ON"

    elif in1 <= il1 and in2 <= il2:
        contract_name = "OFF3"

    elif in1 <= il1 and in2 >= il2:
        contract_name = "OFF1"

    elif in1 >= il1 and in2 <= il2:
        contract_name = "OFF2"

    elif il1 < in1 < ih1 and il2 < in2 < ih2:
        contract_name = "transition"

    else:
        raise ValueError("No current suitable contract")

    # Response parameters from the original AND CSV
    output_concentration = corresponding_y_and(in1, in2, row)

    return contracts[contract_name], contract_name, output_concentration

file_name = "parsed_Eco3C3G3T3.csv"
protein_name = "SicA_InvF"
device_df = pd.read_csv("/home/shrut/bioCircuitProject/library/AND_gate/" + file_name)
row = device_df[device_df["name"] == protein_name].iloc[0]

threshold_df = pd.read_csv("/home/shrut/bioCircuitProject/library/AND_gate/and_thresholds.csv")
threshold_row = threshold_df[(threshold_df["source_file"] == file_name) & (threshold_df["name"] == protein_name)].iloc[0]


# filtering out just the and contracts for the protien output we are looking at
contracts = and_contracts[file_name][protein_name]
u2 = 2.0
u3 = 0
selected_contract, contract_type, output_amount = matching_to_and_contracts(u2, u3, threshold_row, row, contracts)
print (contract_type, output_amount)


0.12180219917159263 of A1_AmtR and the state is: transition
OFF2 0.02
